ML Assignment 2 : ML Models 
Submitted By : Kunjal Chachra , Bits Id. : 2026AJ05392

Step 1 : Understanding the Data 


In [3]:

# Step 1: Import the libraries
import pandas as pd

# Load the dataset
df = pd.read_csv("bank.csv")

# Display the first five rows
print(df.head())

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Check column names and data types
print(df.info())

# Check for missing values
print(df.isnull().sum())

# Check the distribution of the target variable
print(df["deposit"].value_counts())


   age         job  marital  education default  balance housing loan  contact  \
0   59      admin.  married  secondary      no     2343     yes   no  unknown   
1   56      admin.  married  secondary      no       45      no   no  unknown   
2   41  technician  married  secondary      no     1270     yes   no  unknown   
3   55    services  married  secondary      no     2476     yes   no  unknown   
4   54      admin.  married   tertiary      no      184      no   no  unknown   

   day month  duration  campaign  pdays  previous poutcome deposit  
0    5   may      1042         1     -1         0  unknown     yes  
1    5   may      1467         1     -1         0  unknown     yes  
2    5   may      1389         1     -1         0  unknown     yes  
3    5   may       579         1     -1         0  unknown     yes  
4    5   may       673         2     -1         0  unknown     yes  
Dataset shape: (11162, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11

**Conclusion – Dataset Exploration**

The Bank Marketing dataset contains 11,162 instances and 17 columns, consisting of 16 input features and one target variable, `deposit`. The dataset contains seven numerical columns and ten object-type columns, including the target variable. No missing values were detected. The target variable has two classes: 5,873 customers did not subscribe to a term deposit, while 5,289 customers subscribed. Therefore, this dataset is suitable for a binary classification task and satisfies the assignment's minimum requirements of 500 instances and 12 input features.

Step 2 — Data preprocessing

In [4]:

# Step 2: Separate features and target

X = df.drop("deposit", axis=1)
y = df["deposit"].map({"no": 0, "yes": 1})

# Identify numerical and categorical input features
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Input features:", X.shape)
print("Target:", y.shape)
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

# Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Input features: (11162, 16)
Target: (11162,)
Numerical features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Training samples: 8929
Testing samples: 2233


**Conclusion – Feature Selection and Train-Test Split**

The dataset was separated into 16 input features (X) and one target variable (y), where the target values were encoded as 0 for 'no' and 1 for 'yes'. The input features consist of seven numerical features and nine categorical features. The dataset was divided into training and testing sets using an 80:20 ratio, resulting in 8,929 training samples and 2,233 testing samples. Stratified sampling was used to maintain the approximate class distribution in both sets. This separation enables the models to learn from the training data and be evaluated on unseen test data.

Step 3 — Encode categorical features and scale numerical features

In [5]:

# Step 3: Encode categorical features and scale numerical features

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Create the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Learn transformations from training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same transformations to test data
X_test_processed = preprocessor.transform(X_test)

# Display the results
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)


Original training shape: (8929, 16)
Processed training shape: (8929, 51)
Original testing shape: (2233, 16)
Processed testing shape: (2233, 51)


**Conclusion – Feature Encoding and Scaling**

The numerical features were standardized using StandardScaler, while the categorical features were converted into numerical representations using OneHotEncoder. The preprocessing transformations were fitted on the training data and then applied to the testing data to prevent data leakage. As a result, the training dataset was transformed from 8,929 rows and 16 columns into 8,929 rows and 51 columns. Similarly, the testing dataset was transformed from 2,233 rows and 16 columns into 2,233 rows and 51 columns. This prepared the data for training and evaluating the five classification models.

Step 4 — Implement Model 1: Logistic Regression

In [6]:

# Step 4: Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Initialize the model
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model
lr_model.fit(X_train_processed, y_train)

# Predict target classes and probabilities
y_pred_lr = lr_model.predict(X_test_processed)
y_prob_lr = lr_model.predict_proba(X_test_processed)[:, 1]

# Calculate evaluation metrics
lr_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "AUC": roc_auc_score(y_test, y_prob_lr),
    "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
    "Recall": recall_score(y_test, y_pred_lr, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_lr, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred_lr)
}

# Display the results
print("Logistic Regression - Evaluation Results")
for metric, score in lr_metrics.items():
    print(f"{metric}: {score:.4f}")


Logistic Regression - Evaluation Results
Accuracy: 0.8262
AUC: 0.9071
Precision: 0.8278
Recall: 0.7996
F1 Score: 0.8135
MCC: 0.6513


**Conclusion – Logistic Regression**

The Logistic Regression classifier achieved an accuracy of 82.62% and an AUC score of 0.9071 on the test dataset. It obtained a precision of 82.78%, recall of 79.96%, F1 score of 0.8135, and MCC score of 0.6513. These results indicate that the model provides good classification performance and strong class-discrimination ability. Its precision and recall values suggest a reasonably balanced ability to identify customers who subscribe to a bank term deposit. The model's performance will be compared with the other four classifiers to determine the overall best-performing model.

Step 5 — Implement Model 2: Decision Tree Classifier

In [7]:

# Step 5: Decision Tree Classifier

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Initialize the model
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=10
)

# Train the model
dt_model.fit(X_train_processed, y_train)

# Generate predictions
y_pred_dt = dt_model.predict(X_test_processed)
y_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

# Calculate evaluation metrics
dt_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_dt),
    "AUC": roc_auc_score(y_test, y_prob_dt),
    "Precision": precision_score(y_test, y_pred_dt, zero_division=0),
    "Recall": recall_score(y_test, y_pred_dt, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_dt, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred_dt)
}

# Display results
print("Decision Tree - Evaluation Results")
for metric, score in dt_metrics.items():
    print(f"{metric}: {score:.4f}")


Decision Tree - Evaluation Results
Accuracy: 0.8186
AUC: 0.8624
Precision: 0.7822
Recall: 0.8554
F1 Score: 0.8172
MCC: 0.6405


**Conclusion – Decision Tree Classifier**

The Decision Tree classifier achieved an accuracy of 81.86% and an AUC score of 0.8624. Its precision was 78.22%, recall was 85.54%, F1 score was 0.8172, and MCC score was 0.6405. The model demonstrated a strong ability to identify customers who subscribed to a bank term deposit, as reflected by its relatively high recall. However, its precision and AUC were lower than those of Logistic Regression. The results suggest that the Decision Tree identifies more positive cases but produces more false-positive predictions. Its overall performance will be compared with the remaining classifiers.

Step 6 — Implement Model 3: K-Nearest Neighbors (KNN)

In [8]:

# Step 6: K-Nearest Neighbors Classifier

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Initialize the model
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_model.fit(X_train_processed, y_train)

# Generate predictions
y_pred_knn = knn_model.predict(X_test_processed)
y_prob_knn = knn_model.predict_proba(X_test_processed)[:, 1]

# Calculate evaluation metrics
knn_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_knn),
    "AUC": roc_auc_score(y_test, y_prob_knn),
    "Precision": precision_score(y_test, y_pred_knn, zero_division=0),
    "Recall": recall_score(y_test, y_pred_knn, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_knn, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred_knn)
}

# Display results
print("KNN - Evaluation Results")
for metric, score in knn_metrics.items():
    print(f"{metric}: {score:.4f}")


KNN - Evaluation Results
Accuracy: 0.8173
AUC: 0.8796
Precision: 0.8199
Recall: 0.7873
F1 Score: 0.8033
MCC: 0.6333


**Conclusion – K-Nearest Neighbors Classifier**

The K-Nearest Neighbors classifier, configured with five neighbors, achieved an accuracy of 81.73% and an AUC score of 0.8796 on the test dataset. It obtained a precision of 81.99%, recall of 78.73%, F1 score of 0.8033, and MCC score of 0.6333. The results indicate that KNN provides reasonably good classification performance. Its precision is close to that of Logistic Regression, although its accuracy, AUC, recall, F1 score, and MCC are lower than the best corresponding results among the models evaluated so far. Its performance may be influenced by the choice of the number of neighbors and the distance-based nature of the algorithm.

Step 7 — Implement Model 4: Naive Bayes

In [9]:

# Step 7: Gaussian Naive Bayes Classifier

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Initialize the model
nb_model = GaussianNB()

# GaussianNB requires dense input
X_train_dense = X_train_processed.toarray() if hasattr(
    X_train_processed, "toarray"
) else X_train_processed

X_test_dense = X_test_processed.toarray() if hasattr(
    X_test_processed, "toarray"
) else X_test_processed

# Train the model
nb_model.fit(X_train_dense, y_train)

# Generate predictions
y_pred_nb = nb_model.predict(X_test_dense)
y_prob_nb = nb_model.predict_proba(X_test_dense)[:, 1]

# Calculate evaluation metrics
nb_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_nb),
    "AUC": roc_auc_score(y_test, y_prob_nb),
    "Precision": precision_score(y_test, y_pred_nb, zero_division=0),
    "Recall": recall_score(y_test, y_pred_nb, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_nb, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred_nb)
}

# Display results
print("Gaussian Naive Bayes - Evaluation Results")
for metric, score in nb_metrics.items():
    print(f"{metric}: {score:.4f}")


Gaussian Naive Bayes - Evaluation Results
Accuracy: 0.7201
AUC: 0.8042
Precision: 0.7837
Recall: 0.5652
F1 Score: 0.6568
MCC: 0.4472


**Conclusion – Gaussian Naive Bayes Classifier**

The Gaussian Naive Bayes classifier achieved an accuracy of 72.01% and an AUC score of 0.8042. Its precision was 78.37%, recall was 56.52%, F1 score was 0.6568, and MCC score was 0.4472. Although the model demonstrated reasonable class-discrimination ability, its recall was comparatively low, indicating that it failed to identify a substantial proportion of actual subscribers. Its overall performance was weaker than that of Logistic Regression, Decision Tree and KNN on this test split. This may be partly explained by the conditional independence and Gaussian distribution assumptions of the algorithm.

Step 8 — Implement Model 5: Support Vector Machine (SVM)

In [10]:

# Step 8: Support Vector Machine (SVM)

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Initialize the model
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    probability=True,
    random_state=42
)

# Train the model
svm_model.fit(X_train_processed, y_train)

# Generate predictions
y_pred_svm = svm_model.predict(X_test_processed)
y_prob_svm = svm_model.predict_proba(X_test_processed)[:, 1]

# Calculate evaluation metrics
svm_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_svm),
    "AUC": roc_auc_score(y_test, y_prob_svm),
    "Precision": precision_score(y_test, y_pred_svm, zero_division=0),
    "Recall": recall_score(y_test, y_pred_svm, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_svm, zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred_svm)
}

# Display results
print("SVM - Evaluation Results")
for metric, score in svm_metrics.items():
    print(f"{metric}: {score:.4f}")


SVM - Evaluation Results
Accuracy: 0.8554
AUC: 0.9194
Precision: 0.8249
Recall: 0.8819
F1 Score: 0.8524
MCC: 0.7124


**Conclusion – Support Vector Machine**

The Support Vector Machine classifier with an RBF kernel achieved an accuracy of 85.54% and an AUC score of 0.9194 on the test dataset. It obtained a precision of 82.49%, recall of 88.19%, F1 score of 0.8524, and MCC score of 0.7124. Among the five evaluated classifiers, SVM achieved the highest values for accuracy, AUC, recall, F1 score, and MCC. These results indicate that SVM provides strong classification performance and effectively identifies customers who subscribe to a bank term deposit. Therefore, SVM is the best-performing model on the current test split, based on the six evaluation metrics.